# Notebook 3: Conservation Scoring & Structure Mapping

**Author:** Chidera Ibe (coibe2)

## Overview

In this notebook, we quantify evolutionary conservation across the PI3K-alpha (PIK3CA) protein
and map those conservation scores onto the 3D structure from PDB 8TSD, which contains
the RLY-2608 allosteric inhibitor bound to a cryptic pocket.

**Key questions:**
1. How conserved is the cryptic drug-binding pocket compared to the rest of the protein?
2. Which pocket-lining residues are most/least conserved across orthologs?
3. How do the pocket residues vary across PI3K paralogs (alpha, beta, gamma, delta)?

**Sections:**
1. Setup & Load MSA
2. Conservation Scoring
3. Fetch & Parse PDB 8TSD
4. Map Conservation to Structure
5. Conservation Profile at the Pocket
6. Paralog Comparison at the Pocket

---
## Section 1: Setup & Load MSA

We begin by importing our project modules and external libraries, then loading the
multiple sequence alignment (MSA) of PIK3CA orthologs that was constructed in Notebook 2.

In [ ]:
# ---- path setup so we can import from src/ ----
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ---- project imports ----
from src.fetchers import fetch_pdb, fetch_all_paralogs
from src.utils import read_fasta, write_fasta
from src.conservation import conservation_scores
from src.alignment import needleman_wunsch

# ---- standard scientific stack ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import py3Dmol

sns.set_style("whitegrid")
%matplotlib inline

In [ ]:
# ---- load ortholog MSA ----
msa_path = os.path.join(PROJECT_ROOT, "data", "processed", "alignments", "pik3ca_orthologs_msa.fasta")
alignment = read_fasta(msa_path)

seq_names = list(alignment.keys())
num_seqs = len(seq_names)
aln_length = len(list(alignment.values())[0])

print(f"Alignment dimensions: {num_seqs} sequences x {aln_length} columns")
print(f"\nSequence names:")
for name in seq_names:
    print(f"  {name}")

---
## Section 2: Conservation Scoring

We compute Shannon entropy-based conservation scores at each alignment column.
The score is defined as:

$$\text{Conservation} = 1 - \frac{H}{H_{\max}}$$

where $H$ is the Shannon entropy of the amino acid distribution at that column and
$H_{\max} = \log_2(20)$ for 20 standard amino acids.

- Score = 1.0 means perfectly conserved (all species share the same residue).
- Score = 0.0 means maximum entropy (all 20 amino acids equally likely).

We use the human PI3K-alpha sequence as the reference so that scores are mapped
to ungapped residue positions in the human protein.

> **Note:** The alignment keys come from Ensembl protein IDs. We try to auto-detect
> the human sequence by looking for `"homo_sapiens"` or `"human"` or `"PI3Ka"` in
> the key names. If none match, we default to the first sequence. You may need to
> set `HUMAN_KEY` manually below.

In [ ]:
# ---- identify the human reference sequence ----
HUMAN_KEY = None

for key in alignment:
    key_lower = key.lower()
    if "homo_sapiens" in key_lower or "human" in key_lower or "pi3ka" in key_lower:
        HUMAN_KEY = key
        break

# Fallback: use the first sequence
if HUMAN_KEY is None:
    HUMAN_KEY = seq_names[0]
    print(f"WARNING: Could not auto-detect human sequence. Using first sequence: {HUMAN_KEY}")
    print("Set HUMAN_KEY manually if this is incorrect.")
else:
    print(f"Human reference sequence: {HUMAN_KEY}")

# Count ungapped length of the human sequence
human_ungapped_len = len(alignment[HUMAN_KEY].replace("-", ""))
print(f"Human PI3Ka ungapped length: {human_ungapped_len} residues")

In [ ]:
# ---- compute conservation scores ----
scores, positions = conservation_scores(alignment, reference_name=HUMAN_KEY)

print(f"Conservation scores computed for {len(scores)} residue positions")
print(f"Score range: [{scores.min():.4f}, {scores.max():.4f}]")

In [ ]:
# ---- plot conservation across the entire protein ----
fig, ax = plt.subplots(figsize=(16, 5))

# Shade domain regions
ax.axvspan(330, 480, alpha=0.15, color="green", label="C2 domain (~330-480)")
ax.axvspan(700, 1068, alpha=0.15, color="cornflowerblue", label="Kinase domain (~700-1068)")

# Plot conservation scores
ax.plot(positions, scores, color="black", linewidth=0.5, alpha=0.8)

# Mean conservation line
mean_cons = np.mean(scores)
ax.axhline(y=mean_cons, color="red", linestyle="--", linewidth=1, alpha=0.7,
           label=f"Mean = {mean_cons:.3f}")

ax.set_xlabel("Residue position (human PI3Ka)", fontsize=12)
ax.set_ylabel("Conservation score", fontsize=12)
ax.set_title("Per-residue Conservation of PI3K-alpha Across Orthologs", fontsize=14)
ax.set_xlim(1, max(positions))
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ---- summary statistics ----
median_cons = np.median(scores)
frac_highly_conserved = np.mean(scores > 0.9)

print(f"Conservation summary statistics:")
print(f"  Mean conservation score:   {mean_cons:.4f}")
print(f"  Median conservation score: {median_cons:.4f}")
print(f"  Fraction with score > 0.9: {frac_highly_conserved:.4f} ({int(np.sum(scores > 0.9))}/{len(scores)} positions)")

---
## Section 3: Fetch & Parse PDB 8TSD

PDB 8TSD contains the crystal structure of PI3K-alpha (p110-alpha catalytic subunit)
bound to the allosteric inhibitor RLY-2608. We will:

1. Fetch the PDB file using our caching fetcher
2. Parse the structure with Biopython's PDBParser
3. Identify the drug ligand (HETATM records)
4. Use NeighborSearch to find protein residues within 5 Angstroms of the ligand

These "pocket-lining" residues define the cryptic binding site whose conservation
we want to analyze.

In [ ]:
# ---- fetch PDB 8TSD ----
pdb_text = fetch_pdb("8TSD")

# Save to a temp file for Biopython parsing
pdb_path = os.path.join(PROJECT_ROOT, "data", "raw", "structures", "8TSD.pdb")
os.makedirs(os.path.dirname(pdb_path), exist_ok=True)
with open(pdb_path, "w") as f:
    f.write(pdb_text)

print(f"PDB file saved to: {pdb_path}")
print(f"File size: {len(pdb_text):,} characters")

In [ ]:
# ---- parse with Biopython ----
from Bio.PDB import PDBParser, NeighborSearch, Selection

parser = PDBParser(QUIET=True)
structure = parser.get_structure("8TSD", pdb_path)
model = structure[0]

# List chains
print("Chains in 8TSD:")
for chain in model:
    num_residues = sum(1 for r in chain.get_residues() if r.id[0] == " ")
    num_het = sum(1 for r in chain.get_residues() if r.id[0] != " ")
    print(f"  Chain {chain.id}: {num_residues} protein residues, {num_het} HETATM groups")

In [ ]:
# ---- identify the ligand ----
chain_A = model["A"]

# Find all non-standard residues (HETATM) that are not water
het_residues = []
for residue in chain_A.get_residues():
    hetflag = residue.id[0]
    if hetflag not in (" ", "W"):  # not protein and not water
        het_residues.append(residue)

# Also check all chains for HETATM residues
print("Non-water HETATM residues found in the structure:")
print(f"{'Chain':<8} {'ResName':<10} {'ResNum':<10} {'NumAtoms':<10}")
print("-" * 38)
for chain in model:
    for residue in chain.get_residues():
        hetflag = residue.id[0]
        if hetflag not in (" ", "W"):
            resname = residue.get_resname()
            resnum = residue.id[1]
            num_atoms = len(list(residue.get_atoms()))
            print(f"{chain.id:<8} {resname:<10} {resnum:<10} {num_atoms:<10}")

In [ ]:
# ---- find pocket-lining residues within 5A of the ligand ----

# Collect protein atoms from chain A
protein_atoms = [
    a for r in chain_A.get_residues()
    if r.id[0] == " "
    for a in r.get_atoms()
]

# Collect ligand atoms (non-protein, non-water) from chain A
ligand_atoms = [
    a for r in chain_A.get_residues()
    if r.id[0] not in (" ", "W")
    for a in r.get_atoms()
]

print(f"Number of protein atoms in chain A: {len(protein_atoms)}")
print(f"Number of ligand atoms in chain A:  {len(ligand_atoms)}")

# NeighborSearch over protein atoms
ns = NeighborSearch(protein_atoms)

pocket_residues = set()
for la in ligand_atoms:
    nearby = ns.search(la.get_vector().get_array(), 5.0, level="R")
    pocket_residues.update(nearby)

# Sort by residue number
pocket_residues = sorted(pocket_residues, key=lambda r: r.id[1])

print(f"\nPocket-lining residues (within 5A of ligand): {len(pocket_residues)}")
print(f"{'ResNum':<8} {'ResName':<10}")
print("-" * 18)
for res in pocket_residues:
    print(f"{res.id[1]:<8} {res.get_resname():<10}")

In [ ]:
# ---- create a DataFrame of pocket residues ----
from Bio.Data.IUPACData import protein_letters_3to1

def three_to_one(resname):
    """Convert 3-letter amino acid code to 1-letter."""
    return protein_letters_3to1.get(resname.capitalize(), "X")

pocket_data = []
for res in pocket_residues:
    resnum = res.id[1]
    resname_3 = res.get_resname()
    resname_1 = three_to_one(resname_3)
    pocket_data.append({
        "resnum": resnum,
        "resname_3letter": resname_3,
        "resname_1letter": resname_1,
    })

pocket_df = pd.DataFrame(pocket_data)
print(f"Pocket residues DataFrame: {len(pocket_df)} residues")
pocket_df

---
## Section 4: Map Conservation to Structure

Now we combine our conservation scores with the structural data. For each pocket-lining
residue identified above, we look up its conservation score from the ortholog MSA.

We then:
1. Create a bar chart of pocket residue conservation
2. Replace B-factors in the PDB file with conservation scores
3. Visualize the structure with py3Dmol, coloring by conservation

In [ ]:
# ---- build a mapping from residue position -> conservation score ----
pos_to_score = dict(zip(positions, scores))

# Look up conservation for each pocket residue
pocket_df["conservation"] = pocket_df["resnum"].map(pos_to_score)

# Report any residues without a score (might be outside the aligned region)
missing = pocket_df[pocket_df["conservation"].isna()]
if len(missing) > 0:
    print(f"WARNING: {len(missing)} pocket residues have no conservation score (not in alignment):")
    print(missing)
else:
    print("All pocket residues have conservation scores.")

pocket_df

In [ ]:
# ---- bar chart of pocket residue conservation ----
fig, ax = plt.subplots(figsize=(14, 5))

pocket_sorted = pocket_df.dropna(subset=["conservation"]).sort_values("resnum")
x_labels = [f"{row.resname_1letter}{row.resnum}" for _, row in pocket_sorted.iterrows()]
cons_values = pocket_sorted["conservation"].values

# Color by conservation: blue = high (conserved), red = low (variable)
colors = plt.cm.coolwarm_r((cons_values - 0.3) / 0.7)  # normalize to ~[0.3, 1.0] range

bars = ax.bar(range(len(x_labels)), cons_values, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(x_labels, rotation=90, fontsize=9)
ax.set_ylabel("Conservation score", fontsize=12)
ax.set_xlabel("Pocket-lining residue", fontsize=12)
ax.set_title("Conservation of RLY-2608 Pocket-Lining Residues", fontsize=14)
ax.axhline(y=mean_cons, color="gray", linestyle="--", linewidth=1,
           label=f"Protein-wide mean = {mean_cons:.3f}")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ---- replace B-factors with conservation scores ----

def replace_bfactors(pdb_text, scores_by_resnum):
    """
    Replace the B-factor column (columns 61-66) in ATOM records with
    conservation scores for chain A residues.
    
    Parameters
    ----------
    pdb_text : str
        Original PDB file content.
    scores_by_resnum : dict
        Mapping of residue number -> conservation score.
    
    Returns
    -------
    str
        Modified PDB text with conservation scores in the B-factor column.
    """
    lines = []
    for line in pdb_text.split("\n"):
        if line.startswith("ATOM"):
            try:
                resnum = int(line[22:26].strip())
                chain = line[21]
                if chain == "A" and resnum in scores_by_resnum:
                    score = scores_by_resnum[resnum]
                    bfactor_str = f"{score:6.2f}"
                    line = line[:60] + bfactor_str + line[66:]
            except (ValueError, IndexError):
                pass
        lines.append(line)
    return "\n".join(lines)


# Build score mapping for all residues (not just pocket)
scores_by_resnum = {pos: float(score) for pos, score in zip(positions, scores)}

modified_pdb = replace_bfactors(pdb_text, scores_by_resnum)
print(f"Modified PDB text generated ({len(modified_pdb):,} characters)")
print("B-factors now represent conservation scores for chain A.")

In [ ]:
# ---- 3D visualization with py3Dmol ----
# Color the structure by conservation (B-factor): blue = conserved, red = variable

pocket_resi_list = pocket_df["resnum"].tolist()

view = py3Dmol.view(width=800, height=600)
view.addModel(modified_pdb, "pdb")

# Cartoon representation colored by B-factor (conservation)
view.setStyle(
    {"chain": "A"},
    {"cartoon": {
        "colorscheme": {
            "prop": "b",
            "gradient": "rwb",
            "min": 0.3,
            "max": 1.0
        }
    }}
)

# Show pocket residues as sticks
view.addStyle(
    {"resi": pocket_resi_list, "chain": "A"},
    {"stick": {
        "colorscheme": {
            "prop": "b",
            "gradient": "rwb",
            "min": 0.3,
            "max": 1.0
        }
    }}
)

# Show ligand in yellow sticks
view.addStyle(
    {"hetflag": True, "not": {"resn": "HOH"}},
    {"stick": {"color": "yellow"}}
)

# Add labels for a few key pocket residues (first, last, and middle)
if len(pocket_resi_list) > 0:
    label_resis = pocket_resi_list[::max(1, len(pocket_resi_list)//5)]  # ~5 evenly spaced labels
    for resi in label_resis:
        row = pocket_df[pocket_df["resnum"] == resi].iloc[0]
        label_text = f"{row['resname_1letter']}{resi}"
        view.addLabel(
            label_text,
            {"fontSize": 10, "fontColor": "white", "backgroundColor": "black",
             "backgroundOpacity": 0.6},
            {"resi": resi, "chain": "A", "atom": "CA"}
        )

view.zoomTo({"chain": "A"})
view.show()

**Interpretation:** In the 3D view above, blue regions are highly conserved across
orthologs and red regions are variable. The RLY-2608 ligand is shown in yellow sticks.
Pocket-lining residues are rendered as sticks so their conservation coloring is visible.

---
## Section 5: Conservation Profile at the Pocket

We now compare the conservation at the drug-binding pocket to the protein-wide
conservation distribution. A pocket that is significantly more conserved than
average suggests functional importance; a less conserved pocket may indicate
that the binding site is more evolutionarily flexible, which has implications
for drug resistance.

In [ ]:
# ---- highlight pocket residues on the full-protein conservation plot ----
fig, ax = plt.subplots(figsize=(16, 5))

# Shade domain regions
ax.axvspan(330, 480, alpha=0.15, color="green", label="C2 domain")
ax.axvspan(700, 1068, alpha=0.15, color="cornflowerblue", label="Kinase domain")

# Plot full conservation trace
ax.plot(positions, scores, color="black", linewidth=0.5, alpha=0.6)

# Highlight pocket residues with red vertical lines and dots
pocket_positions = pocket_df.dropna(subset=["conservation"])
for _, row in pocket_positions.iterrows():
    ax.axvline(x=row["resnum"], color="red", alpha=0.3, linewidth=0.8)

ax.scatter(
    pocket_positions["resnum"],
    pocket_positions["conservation"],
    color="red", s=30, zorder=5, label="Pocket residues"
)

# Mean lines
pocket_mean = pocket_positions["conservation"].mean()
ax.axhline(y=mean_cons, color="gray", linestyle="--", linewidth=1,
           label=f"Protein-wide mean = {mean_cons:.3f}")
ax.axhline(y=pocket_mean, color="red", linestyle="--", linewidth=1,
           label=f"Pocket mean = {pocket_mean:.3f}")

ax.set_xlabel("Residue position", fontsize=12)
ax.set_ylabel("Conservation score", fontsize=12)
ax.set_title("Pocket Residues Highlighted on Full Conservation Profile", fontsize=14)
ax.set_xlim(1, max(positions))
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ---- compare pocket vs. genome-wide conservation ----
print(f"Conservation comparison:")
print(f"  Protein-wide mean conservation: {mean_cons:.4f}")
print(f"  Pocket mean conservation:       {pocket_mean:.4f}")
print(f"  Difference (pocket - overall):  {pocket_mean - mean_cons:+.4f}")
print()

if pocket_mean > mean_cons:
    print("The pocket is MORE conserved than the protein average.")
    print("This suggests the cryptic binding site overlaps with a functionally important region.")
else:
    print("The pocket is LESS conserved than the protein average.")
    print("This suggests the binding site may be in an evolutionarily flexible region.")

In [ ]:
# ---- formatted table of pocket residues sorted by conservation ----
pocket_sorted_by_score = pocket_df.dropna(subset=["conservation"]).sort_values(
    "conservation", ascending=True
).reset_index(drop=True)

print(f"Pocket residues sorted by conservation score (ascending):")
print(f"{'Rank':<6} {'Residue':<10} {'Position':<10} {'Conservation':<14} {'Category'}")
print("-" * 55)
for i, (_, row) in enumerate(pocket_sorted_by_score.iterrows(), 1):
    cat = "CONSERVED" if row["conservation"] >= 0.9 else (
        "moderate" if row["conservation"] >= 0.7 else "variable"
    )
    print(f"{i:<6} {row['resname_1letter']}{int(row['resnum']):<9} {int(row['resnum']):<10} {row['conservation']:<14.4f} {cat}")

---
## Section 6: Paralog Comparison at the Pocket

PI3K has four class I catalytic isoforms: alpha (PIK3CA), beta (PIK3CB),
gamma (PIK3CG), and delta (PIK3CD). Understanding which pocket residues
are conserved vs. divergent across paralogs is critical for designing
**isoform-selective** inhibitors.

Residues where alpha differs from ALL other isoforms are **selectivity-determining
residues** -- they are the positions a drug can exploit for alpha-specificity.

In [ ]:
# ---- load paralog alignment ----
paralog_msa_path = os.path.join(
    PROJECT_ROOT, "data", "processed", "alignments", "pik3_paralogs_msa.fasta"
)
paralog_aln = read_fasta(paralog_msa_path)

print(f"Paralog alignment: {len(paralog_aln)} sequences")
for name in paralog_aln:
    ungapped = len(paralog_aln[name].replace('-', ''))
    print(f"  {name}: {ungapped} residues (ungapped)")

In [ ]:
# ---- identify paralog keys ----
# We expect keys like PI3Ka_PIK3CA, PI3Kb_PIK3CB, PI3Kg_PIK3CG, PI3Kd_PIK3CD
paralog_keys = list(paralog_aln.keys())

# Auto-detect alpha key
alpha_key = None
beta_key = None
gamma_key = None
delta_key = None

for key in paralog_keys:
    kl = key.lower()
    if "alpha" in kl or "pi3ka" in kl or "pik3ca" in kl:
        alpha_key = key
    elif "beta" in kl or "pi3kb" in kl or "pik3cb" in kl:
        beta_key = key
    elif "gamma" in kl or "pi3kg" in kl or "pik3cg" in kl:
        gamma_key = key
    elif "delta" in kl or "pi3kd" in kl or "pik3cd" in kl:
        delta_key = key

print(f"Alpha key: {alpha_key}")
print(f"Beta key:  {beta_key}")
print(f"Gamma key: {gamma_key}")
print(f"Delta key: {delta_key}")

if alpha_key is None:
    print("\nWARNING: Could not auto-detect alpha isoform key.")
    print("Defaulting to first key. Please set alpha_key manually if needed.")
    alpha_key = paralog_keys[0]

In [ ]:
# ---- map alignment columns to alpha residue positions ----
# Build a mapping: alpha_resnum -> alignment column index
alpha_seq = paralog_aln[alpha_key]
alpha_resnum_to_col = {}  # resnum (1-based) -> column index
alpha_col_to_resnum = {}  # column index -> resnum

resnum = 0
for col_idx, aa in enumerate(alpha_seq):
    if aa != "-":
        resnum += 1
        alpha_resnum_to_col[resnum] = col_idx
        alpha_col_to_resnum[col_idx] = resnum

print(f"Alpha sequence has {resnum} ungapped positions in the paralog alignment")

In [ ]:
# ---- extract amino acids at pocket positions for each isoform ----
isoform_keys = {
    "alpha": alpha_key,
    "beta": beta_key,
    "gamma": gamma_key,
    "delta": delta_key,
}

paralog_comparison = []

for _, row in pocket_df.iterrows():
    resnum = int(row["resnum"])
    
    if resnum not in alpha_resnum_to_col:
        continue  # skip if not in alignment
    
    col_idx = alpha_resnum_to_col[resnum]
    
    entry = {"position": resnum, "alpha_AA": alpha_seq[col_idx]}
    
    all_match_alpha = True
    for iso_name, iso_key in isoform_keys.items():
        if iso_key is None or iso_name == "alpha":
            continue
        iso_seq = paralog_aln[iso_key]
        iso_aa = iso_seq[col_idx] if col_idx < len(iso_seq) else "-"
        entry[f"{iso_name}_AA"] = iso_aa
        if iso_aa != entry["alpha_AA"]:
            all_match_alpha = False
    
    entry["is_conserved_across_isoforms"] = all_match_alpha
    paralog_comparison.append(entry)

paralog_df = pd.DataFrame(paralog_comparison)
print(f"Paralog comparison for {len(paralog_df)} pocket residues:")
paralog_df

In [ ]:
# ---- identify selectivity-determining residues ----
# Positions where alpha differs from ALL other isoforms
selectivity_residues = []

for _, row in paralog_df.iterrows():
    alpha_aa = row["alpha_AA"]
    other_aas = []
    for col in ["beta_AA", "gamma_AA", "delta_AA"]:
        if col in row and pd.notna(row[col]):
            other_aas.append(row[col])
    
    if len(other_aas) > 0 and all(aa != alpha_aa for aa in other_aas):
        selectivity_residues.append(int(row["position"]))

if selectivity_residues:
    print(f"Selectivity-determining residues (alpha differs from ALL other isoforms):")
    for resnum in selectivity_residues:
        row = paralog_df[paralog_df["position"] == resnum].iloc[0]
        parts = [f"alpha={row['alpha_AA']}"]
        for iso in ["beta", "gamma", "delta"]:
            col = f"{iso}_AA"
            if col in row:
                parts.append(f"{iso}={row[col]}")
        print(f"  Position {resnum}: {', '.join(parts)}")
else:
    print("No positions found where alpha differs from ALL other isoforms.")
    print("Showing positions where alpha differs from at least one isoform:")
    variable = paralog_df[~paralog_df["is_conserved_across_isoforms"]]
    for _, row in variable.iterrows():
        parts = [f"alpha={row['alpha_AA']}"]
        for iso in ["beta", "gamma", "delta"]:
            col = f"{iso}_AA"
            if col in row:
                parts.append(f"{iso}={row[col]}")
        print(f"  Position {int(row['position'])}: {', '.join(parts)}")

In [ ]:
# ---- heatmap: pocket residues x isoforms, colored by match to alpha ----
iso_columns = [c for c in ["alpha_AA", "beta_AA", "gamma_AA", "delta_AA"] if c in paralog_df.columns]

# Build a binary matrix: 1 = matches alpha, 0 = different
heatmap_data = []
for _, row in paralog_df.iterrows():
    alpha_aa = row["alpha_AA"]
    matches = []
    for col in iso_columns:
        if pd.notna(row[col]):
            matches.append(1 if row[col] == alpha_aa else 0)
        else:
            matches.append(np.nan)
    heatmap_data.append(matches)

heatmap_matrix = pd.DataFrame(
    heatmap_data,
    index=[f"{row['alpha_AA']}{int(row['position'])}" for _, row in paralog_df.iterrows()],
    columns=[c.replace("_AA", "") for c in iso_columns]
)

fig, ax = plt.subplots(figsize=(6, max(8, len(paralog_df) * 0.35)))

# Create an annotation matrix showing the actual amino acid
annot_data = []
for _, row in paralog_df.iterrows():
    annot_row = []
    for col in iso_columns:
        if col in row and pd.notna(row[col]):
            annot_row.append(str(row[col]))
        else:
            annot_row.append("-")
    annot_data.append(annot_row)

annot_matrix = pd.DataFrame(
    annot_data,
    index=heatmap_matrix.index,
    columns=heatmap_matrix.columns
)

sns.heatmap(
    heatmap_matrix,
    annot=annot_matrix,
    fmt="s",
    cmap=["salmon", "lightblue"],
    cbar_kws={"label": "Matches alpha (1=yes, 0=no)"},
    linewidths=0.5,
    linecolor="gray",
    ax=ax,
    vmin=0,
    vmax=1
)

ax.set_title("Pocket Residue Identity Across PI3K Isoforms", fontsize=13)
ax.set_xlabel("Isoform", fontsize=11)
ax.set_ylabel("Pocket residue (alpha numbering)", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ---- save results ----
results_dir = os.path.join(PROJECT_ROOT, "data", "results")
os.makedirs(results_dir, exist_ok=True)

# Save conservation scores
conservation_df = pd.DataFrame({
    "position": positions,
    "conservation_score": scores
})
cons_out_path = os.path.join(results_dir, "conservation_scores.csv")
conservation_df.to_csv(cons_out_path, index=False)
print(f"Conservation scores saved to: {cons_out_path}")

# Save pocket analysis
pocket_out_path = os.path.join(results_dir, "pocket_residues.csv")
pocket_df.to_csv(pocket_out_path, index=False)
print(f"Pocket residues saved to:     {pocket_out_path}")

print(f"\nConservation scores: {len(conservation_df)} positions")
print(f"Pocket residues:     {len(pocket_df)} residues")

---
## Summary

In this notebook we:

1. **Computed per-residue conservation scores** from the ortholog MSA using Shannon entropy,
   mapped to human PI3K-alpha residue positions.

2. **Identified the RLY-2608 binding pocket** in PDB 8TSD using a 5-Angstrom neighbor
   search around the ligand atoms.

3. **Mapped conservation onto the 3D structure** by replacing B-factors with conservation
   scores and visualizing with py3Dmol.

4. **Compared pocket conservation** to the protein-wide average, revealing whether the
   cryptic binding site is in a conserved or variable region.

5. **Analyzed paralog variation** at the pocket, identifying selectivity-determining
   residues that could be exploited for alpha-specific drug design.

**Next:** In Notebook 4, we will perform deeper evolutionary analysis of the pocket,
including phylogenetic context and positive selection analysis.